## GPT2
### Decoder-Only

### 1.导入一些相关的包/库



In [42]:
import torch                        # PyTorch 的核心库，包含张量（Tensor）操作
import torch.nn as nn               # Neural Network (神经网络) 模块，包含各种层（如全连接层、卷积层）
import torch.nn.functional as F     # 函数式接口，包含激活函数、损失函数等无参数的计算
from torch.utils.data import Dataset    # 数据集基类，用于定义"怎么读单个数据"
from torch.utils.data import DataLoader # 数据加载器，用于定义"怎么把数据打包(batch)喂给模型"
from dataclasses import dataclass   # Python 标准库，用于快速创建只包含数据的类（常用作配置参数）
import math                         # Python 标准库，包含数学运算函数
# 设置 CPU 生成随机数的种子，保证实验结果可复现
torch.manual_seed(1024)

### 2.定义一些GPT参数



In [43]:

@dataclass
class GPTConfig:
    # -----------------------------------------------------------
    # 1. 记忆力 (Context Window)
    # -----------------------------------------------------------
    block_size: int = 128
    # 含义：上下文窗口大小 / 序列最大长度
    # 通俗理解：模型一眼能看多少个字。
    # 详解：模型在预测下一个字时，最多只能回头看前 512 个字。
    #      超过这个长度，前面的内容它就"忘"了。
    #      (比如你在写第 513 个字时，它已经看不见第 1 个字了)。

    # -----------------------------------------------------------
    # 2. 学习速度/并行能力 (Parallelism)
    # -----------------------------------------------------------
    batch_size: int = 8
    # 含义：批次大小
    # 通俗理解：老师一次批改几本作业。
    # 详解：模型训练不是一句话一句话学的，而是把 12 句话捆在一起，
    #      并行地喂给 GPU 计算。这个数字越大，训练越快（但也越吃显存）。

    # -----------------------------------------------------------
    # 3. 大脑深度 (Depth)
    # -----------------------------------------------------------
    n_layer: int = 2
    # 含义：Transformer Block 的层数
    # 通俗理解：模型有多少层"过滤器"或者"专家"。
    # 详解：数据进入模型后，要经过 12 道工序的处理。
    #      层数越深，模型逻辑推理能力越强，能理解更抽象的概念。
    #      (GPT-2 Small 就是 12 层)。

    # -----------------------------------------------------------
    # 4. 思考角度 (Attention Heads)
    # -----------------------------------------------------------
    n_head: int = 4
    # 含义：多头注意力的头数
    # 通俗理解：模型在读一句话时，有多少个"心眼"同时在看。
    # 详解：比如读"苹果"这个词：
    #      第1个头关注它的颜色，第2个头关注它的形状，第3个头关注它是水果...
    #      12个头就代表它能从 12 个不同的角度去理解词与词之间的关系。

    # -----------------------------------------------------------
    # 5. 词汇理解力 (Embedding Dimension)
    # -----------------------------------------------------------
    n_embd: int = 256
    # 含义：嵌入维度 / 隐藏层大小
    # 通俗理解：用来描述一个词的"特征向量"有多长。
    # 详解：在计算机眼里，"猫"这个字不是汉字，而是一串数字。
    #      这里规定用 768 个数字来描述"猫"。
    #      数字越多（维度越高），能包含的信息就越丰富，描述得越精准。
    #      (注：通常 n_embd 除以 n_head 必须是整数，这里 768/12 = 64)。
    
    hidden_dim : int = n_embd
    # 含义：前馈神经网络的隐藏层维度
    # 通俗理解：模型在每个位置上"思考"时，能"思考"出多少个可能的答案。
    # 详解：在 Transformer 的每个位置上，都有一个小型的前馈神经网络。
    #      这个网络的隐藏层维度是 768，意味着它能"思考"出 768 个不同的可能性。
    #      维度越高，模型的表达能力越强，但计算量也越大。
    # 隐藏层是什么
    # 隐藏层是神经网络中的一个概念，位于输入层和输出层之间。
    # 它接收前一层的所有输入数据，并通过一系列的数学运算（如加权求和、激活函数等）处理这些数据，然后产生下一层的输入。
    # 在Transformer模型中，每个位置的前馈神经网络的隐藏层维度通常与嵌入维度相同，即768维。这有助于保持信息的一致性和模型的表达能力。


    # ------------------------------------------
    # 6. 防作弊机制 (Regularization)
    # -----------------------------------------------------------
    dropout: float = 0.1
    # 含义：丢弃率 (10%)
    # 通俗理解：故意让模型"失忆"或者"脑神经断连"一下。
    # 详解：在训练时，随机把 10% 的神经元关掉（置为0），不让它们工作。
    #      为什么要这么做？为了防止模型"死记硬背"（过拟合）。
    #      这逼迫模型必须学会举一反三，而不是只靠某几个神经元记答案。

    # -----------------------------------------------------------
    # 7. 每个头的视野 (Head Size)
    # -----------------------------------------------------------
    head_size: int = n_embd // n_head
    # 含义：每个注意力头的维度
    # 通俗理解：每个"心眼"能看的信息量。
    # 注意力头会做什么操作，怎么样用到head_size
    # 注意力头会根据输入的词向量，计算出每个词与其他词之间的相关性（注意力分数）。
    # 然后，它会根据这些分数，动态地调整每个词的表示，使得模型能够更好地理解词与词之间的关系。
    # head_size 决定了每个注意力头在计算注意力分数时所使用的向量维度。
    # head_size 是一个词一个词过的吗？
    # head_size 不是一个词一个词过的，而是每个注意力头在处理整个输入序列时使用的向量维度。
    # 输入序列有本质是什么
    # 输入序列本质上是一个由多个词组成的列表，每个词都被表示为一个向量（词嵌入）。
    # 详解：因为总的嵌入维度是 768，而注意力头有 12 个，
    #      所以每个头分到的维度就是 768/12=64。
    # 为什么是n_embd // n_head
    # 因为每个注意力头需要均分总的嵌入维度。
    # 数学本质：

    # -----------------------------------------------------------
    # 8. 词汇表大小
    # -----------------------------------------------------------
    #gpt2官方的tokenizer
    vocab_size: int = 50257
    # 含义：词汇表大小
    # 通俗理解：模型能认识多少个不同的词/符号。
    # 详解：GPT-2 使用的 BPE 分词器一共包含 50257 个不同的 token。
    #      这些 token 包括常见的汉字、字母、标点符号，甚至一些特殊符号。    


## GPT的结构
### 1. SingleHeadAttention



In [44]:
class singleHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        # 生成 Key 矩阵
        self.key = nn.Linear(config.n_embd, config.head_size)
        # key是什么？
        # 在计算注意力分数时，模型会将输入的嵌入向量通过线性变换生成 Key 矩阵。
        # 这个矩阵帮助模型理解输入序列中各个位置的信息，从而决定在计算注意力时应该关注哪些部分。
        # key用来做什么？
        # Key 矩阵用于计算注意力分数，帮助模型确定在处理当前输入时，应该关注输入序列中的哪些位置。
        # 通过与 Query 矩阵进行点积运算，模型可以计算出每个位置的重要性，从而动态地调整对不同位置的关注度。
        # nn.Linear(config.n_embd, config.head_size) 的含义
        # 这表示一个线性变换层，将输入的嵌入向量从 config.n_embd 维度映射到 config.head_size 维度。
        # config.n_embd: 嵌入维度，表示每个词的特征向量长度。
        # config.head_size: 每个注意力头的维度，表示该头能够处理的信息量。
        # 通过这个线性层，模型可以学习到如何将输入的高维信息压缩或转换为适合该注意力头处理的低维表示
        # 目的：为了让模型能够有效地计算注意力分数，从而更好地理解输入序列中的信息分布。
        self.query = nn.Linear(config.n_embd, config.head_size)
        #query是什么？
        # Query 矩阵是注意力机制中的另一个关键组成部分。
        # 它通过线性变换将输入的嵌入向量转换为 Query 矩阵。
        # query用来做什么？
        # Query 矩阵用于与 Key 矩阵进行点积运算，以计算注意力分数。
        # 这些分数反映了当前输入与输入序列中各个位置之间的相关性，帮助模型决定应该关注哪些部分的信息。

        self.value = nn.Linear(config.n_embd, config.head_size)
        # nn.Linear(config.n_embd, config.head_size) 的返回的是Callable Object，就是一个函数
        self.head_size = config.head_size

        # attention_mask的新写法：通过register_buffer注册
        # 因为不用计算梯度
        # mask的作用：屏蔽掉未来的信息，只关注过去的信息
        # 2. 这个 Mask 是用来做什么的？
        # 核心目的：防止“剧透” (Prevent Peeking / Causal Masking)

        # 在训练 GPT 这种生成式模型时，我们是把整句话一次性喂进去的。比如句子是 "I love AI"。 如果不加限制，Self-Attention 机制会让每个词都能看到所有其他的词。

        # 当模型在处理 "I" 的时候，它能看到后面的 "love" 和 "AI"。

        # 这就作弊了！因为在预测时，模型说完 "I" 之后，是不应该知道后面是 "love" 的。
        

        self.register_buffer(
            'attention_mask',
            #tril是下三角矩阵·
            # block_size 是文本的最大长度

            # 保留主对角线及以下的数值，把上面的全部变成 0
            torch.tril(
                # 创建一个全 1 的矩阵
                torch.ones((config.block_size, config.block_size))
            )
        )

        #dropout是什么: 随机失忆
        # 为什么要有dropout: 解决过拟合
        # Dropout的数学底层原理
        # Dropout 是一种正则化技术，通过在训练过程中随机"丢弃"（置为零）一部分神经元的输出，来防止模型过拟合。
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, hidden_dim = x.size()
        # batch_size: 批次大小
        # seq_len: 序列长度
        # hidden_dim: 隐藏层维度 (嵌入维度)
        
        # 1. 生成 Query, Key, Value
        # [Batch, Seq, Dim] -> [Batch, Seq, Head_Size]
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        # PyTorch 的矩阵乘法 (torch.matmul 或 @) 极其“死板”，它只认最后两个维度做乘法
        # 也就是把 [Batch, Seq, Head_Size] -> [Batch, Head_Size, Seq]
        weight = q @ k.transpose(-2, -1)
        # -2 和 -1 分别表示倒数第二个和倒数第一个维度
        # 就是倒数第二个和倒数第一个维度转置吗？
        # weights 的形状是 (batch_size, seq_len, seq_len)
        weight = weight.masked_fill(
            # [:seq_len, :seq_len] 是因为实际的序列长度可能小于 block_size
            # 意为着只取前 seq_len 行和前 seq_len 列，其余位置填充为 -inf
            self.attention_mask[:seq_len, :seq_len] == 0, 
            float('-inf')
            )

        #记得weight除以sqrt(d_k)
        # 为什么要除以这个 8？（为什么要缩放？）
        #这是为了救 Softmax 一命，防止梯度消失。
        weight = weight / math.sqrt(self.head_size)
        weight = F.softmax(weight, dim=-1)

        #Attention Dropout要放在weight权重化前面
        weight = self.dropout(weight)
        output = weight @ v  # [Batch, Seq, Head_Size]
        return output



### 2.MutiHeadAttention



In [45]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                singleHeadAttention(config) 
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor):
        # x: [Batch, Seq, Dim]
        output = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )  # [Batch, Seq, Dim]
        output = self.proj(output)
        output = self.dropout(output)
        return output

### 3.FeedForward(MLP)



In [46]:
class FeedForward(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.hidden_dim, 4 * config.hidden_dim),
            nn.GELU(),
            nn.Linear(4 * config.hidden_dim, config.hidden_dim),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)

### 4.Block

##### 它对应的是原始 Transformer 论文中的 Decoder Layer（解码器层），但针对 GPT-2 做了一些关键的结构调整（主要是 LayerNorm 的位置）。

###### 数据流向如下（代码逻辑）：

- 输入 x
- 分支 1 (Attention)：
  - 先过 LayerNorm: ln1(x)
  - 再过 Attention: attn(ln1(x))
  - 残差连接（加回原输入）: x = x + attn(ln1(x))
- 分支 2 (MLP)：
  - 先过 LayerNorm: ln2(x)
  - 再过 MLP: mlp(ln2(x))
  - 残差连接（加回原输入）: x = x + mlp(ln2(x))
- 输出 x

#### 残差连接 (Residual Connection)

- 什么是残差连接？
- 残差连接是一种神经网络结构设计技巧，旨在解决深层网络训练过程中的梯度消失或爆炸问题。
- 它通过在网络层之间添加直接的跳跃连接，使得输入可以绕过某些层，直接传递到后续层。
- 公式上表示为：y = F(x) + x，其中 F(x) 是经过若干层处理后的输出，x 是输入。


In [47]:
class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.att = MultiHeadAttention(config) #MHA: Multi-Head Attention
        self.ffn = FeedForward(config) #FFN: Feed-Forward Network
        self.ln1 = nn.LayerNorm(config.hidden_dim) #Layer Normalization
        self.ln2 = nn.LayerNorm(config.hidden_dim) #Layer Normalization
    
    def forward(self, x: torch.Tensor):
        # x: [Batch, Seq, Dim]
        
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

### 5. GPT的完整架构
- emebdedding层：将输入的文本转换为向量表示
- position层：处理文本的位置信息
- - position embedding 从 0，1， xxx embedding 升级到了 rope embedding，解决了位置编码的局限性
- norm层：对前一层的输出进行标准化处理，使其更加稳定
- - 从layer norm 升级到了 RMS norm

- mlp层：多层感知机，用于处理前一层的输出
- - mlp 升级到了swigule
- - mha 升级到了 gqa
- block层：由多个attention模块和mlp模块组成，用于处理前一层的输出

In [48]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        #embedding层
        # 这里采用绝对位置编码
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        # position层
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        #block_size = 512
        self.blocks = nn.Sequential(
            *[ Block(config) for _ in range(config.n_layer)]
        )
        # *是在解包列表/元组/可迭代对象
        # 最后的layer norm层
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias = False)
        # 为什么输出是vocab_size，因为要预测下一个词是什么
        # 现在的SLM（Softmax Language Model）模型会用tie weight来减少参数量
        # tie weight的原理是输入的embedding矩阵和输出的head矩阵共享参数
        # 也就是说head的weight矩阵是token_embedding的weight矩阵
        # linear (4 -> 8), weight的shape实际上是 8 * 4
        #非常重要的一点
        self.token_embedding_table.weight = self.lm_head.weight

    #初始化为高斯分布(正态分布)
    def _init_weight(self, module):
        #linear是有bias的
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

        #embedding是没有bias的

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        #idx输入是token id
        #idx的shape是(batch_size, seq_len)
        # targets是目标的token id
        # 所以shape要一样是(batch_size, seq_len)
        batch_size, seq_len = idx.size()
        #batch是batch_size,本质是样本数
        #seq_len是序列长度，本质是文本长度
        token_embed =  self.token_embedding_table(idx) # (batch_size, seq_len, n_embd)
        position_embed = self.position_embedding_table(
            # 要确保位置编码和输入的idx同一个设备
            # 目的：避免设备不匹配错误，比如一个在CPU上，一个在GPU上
            torch.arange(seq_len, device=idx.device)
        )
        # 经典题目：token_embed 和 position_embed 为什么可以相加？
        # 因为它们的形状是一样的，都是 (batch_size, seq_len, n_embd)
        # 绝对位置编码
        x = token_embed + position_embed  # (batch_size, seq_len, n_embd)
        x = self.blocks(x)  # (batch_size, seq_len, n_embd）
        x = self.ln_final(x)  # (batch_size, seq_len, n_embd)
        logits = self.lm_head(x)  # (batch_size, seq_len, vocab_size)


        # 如果targets是None，说明是预测模式
        if targets is None:
            loss = None
            # 预测模式
        # 如果targets不是None，说明是训练模式
        # 计算损失
        # 训练模式
        else:
            # 计算交叉熵损失
            # logits: (batch_size, seq_len, vocab_size)
            # targets: (batch_size, seq_len)
            # PyTorch 的 CrossEntropyLoss 要求输入是 (N, C) 形状
            # 其中 N 是样本数，C 是类别数
            # 所以我们需要把 logits 和 targets 展平
            batch_size, seq_len, vocab_size = logits.size() 
            # (batch_size, seq_len, vocab_size) -> (batch_size * seq_len, vocab_size)
            # 使用新变量 logits_flat，保持原始 logits 的 3D 形状不变
            logits_flat = logits.view(batch_size * seq_len, vocab_size)
            # targets: (batch_size, seq_len) -> (batch_size * seq_len)
            targets_flat = targets.view(batch_size * seq_len)
            # 用展平后的变量计算损失
            loss = F.cross_entropy(logits_flat, targets_flat)
            
        return logits, loss
    
    def generate(self, idx: torch.Tensor, max_new_tokens: int):
        # idx: (batch_size, seq_len) 初始的 token 序列
        # max_new_tokens: 要生成的新 token 数量
        
        for _ in range(max_new_tokens):
            # 如果序列长度超过 block_size，只保留最后 block_size 个 token
            idx_cond = idx if idx.size(1) <= self.position_embedding_table.num_embeddings else idx[:, -self.position_embedding_table.num_embeddings:]
            
            # 前向传播获取 logits
            logits, _ = self(idx_cond)  # (batch_size, seq_len, vocab_size)
            
            # 只取最后一个时间步的 logits
            logits = logits[:, -1, :]  # (batch_size, vocab_size)
            
            # 转换为概率分布
            probs = F.softmax(logits, dim=-1)  # (batch_size, vocab_size)
            
            # 从概率分布中采样下一个 token
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
            
            # 将新 token 追加到序列后面
            idx = torch.cat([idx, idx_next], dim=1)  # (batch_size, seq_len + 1)
        
        return idx




### 6.导入数据
#### 了解输入是长什么样的

In [49]:
# Dataset是torch.utils.data中的类，用于自定义数据集
class mydataset(Dataset):
    def __init__(self, path, black_size = 512):
        # tiktoken是OpenAI开源的一个tokenizer库
        import tiktoken
        self.encoder = tiktoken.get_encoding("gpt2")
        self.block_size = black_size

        self.encoded_data = [] # 存储编码后的数据

        # 特殊符号 标记文本结束
        # <|endoftext|> 本质是[50256]，是GPT-2中表示文本结束的特殊token
        #eos是end of string的意思，表示字符串的结束
        # encoder.encode是将字符串转换为token的列表
        # allow_special_tokens参数表示是否允许特殊符号被编码
        self.eos_token = self.encoder.encode(
            "<|endoftext|>",
            allowed_special =  {"<|endoftext|>"}
        )[0]

        # 读取数据
        import json
        self.max_lines = 1000
        raw_data = []
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= self.max_lines:
                    break
                try:
                    # 读取每一行的文本内容 
                    # loads是将字符串转换为字典 
                    # "text"字典键对应的值是文本内容
                    text = json.loads(line.strip())["text"]
                    raw_data.append(text)
                except Exception as e:
                    continue
        
        # 对文本进行编码 并截断到指定长度
        full_encoded_data = []
        for text in raw_data:
            # 把字符串变成 token id 序列
            encoded_text = self.encoder.encode(text) #返回的是一个list
            # self.eos_token是一个int [50256]
            # 在每个文本后面加上eos_token
            # full_encoded_data存储了所有文本的编码结果，每个文本后面都加上eos_token
            full_encoded_data.extend(encoded_text + [self.eos_token])

            # 将长文本分割成多个block_size长度的片段
        for i in range(0, len(full_encoded_data), self.block_size):
            # 要取512 + 1个token，因为要预测下一个token
            chunk = full_encoded_data[i:i + self.block_size + 1]
            # 特殊情况处理：如果最后一块不够长，做padding
            # padding到block_size + 1长度 512 + 1 = 513 
            if len(chunk) < self.block_size + 1:
                chunk += [self.eos_token] * (self.block_size + 1 - len(chunk))
            #存拆分且编码后的数据块
            self.encoded_data.append(chunk)

    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        # 输入是前block_size个token
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        # 目标是后block_size个token
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y
        
    def encode(self, text):
        return self.encoder.encode(text)
    
    def decode(self, token_ids):
        return self.encoder.decode(token_ids)






### 7.下载小样本语料
使用 Hugging Face `datasets` 拉取 `mobvoi/seq-monkey-general-open-corpus` 的前 2% 保存到本地 `data/monkey_sample.jsonl`。

In [50]:
# 下载并保存开源文本小样本，包含多个兜底数据集
# 优先尝试 mobvoi/seq-monkey-general-open-corpus，不可用则换其他公开语料
from pathlib import Path
from datasets import load_dataset

out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "monkey_sample.jsonl"

# 如果已有文件则直接跳过，避免重复下载/覆盖
if out_path.exists():
    print(f"found existing {out_path}, skip download")
else:
    # 候选数据集（按优先级）
    candidates = [
        {"path": "mobvoi/seq-monkey-general-open-corpus", "config": None, "split": "train[:5%]"},
        {"path": "shibing624/baike2018qa", "config": None, "split": "train[:2%]"},  # 中文百科 QA，小体量
        {"path": "wikitext", "config": "wikitext-2-raw-v1", "split": "train[:5%]"},  # 英文兜底，体量小
    ]

    used = None
    ds = None
    for cand in candidates:
        try:
            args = [cand["path"]]
            if cand["config"]:
                args.append(cand["config"])
            ds = load_dataset(*args, split=cand["split"])
            used = cand
            print(f"loaded {cand['path']} split {cand['split']}")
            break
        except Exception as e:
            print(f"skip {cand['path']}: {e}")
            continue

    if ds is None:
        raise RuntimeError("无法加载任何候选数据集，请检查网络或更换数据集")

    # 可选：额外再裁剪，例如只取前 8000 条
    # ds = ds.select(range(8000))

    if len(ds) > 0:
        ds.to_json(out_path.as_posix(), orient="records", lines=True)
        print(f"saved {len(ds)} rows to {out_path} (source={used['path']})")
    else:
        print("dataset empty; nothing written")

found existing data/monkey_sample.jsonl, skip download


### 8.运行相关函数

In [51]:
model = GPT(GPTConfig())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

#打印模型参数量

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params / 1e6:.2f} M")
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# 设置学习率
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
# 加载数据集
dataset = mydataset("data/monkey_sample.jsonl", black_size=128)

# 计算训练集和验证集的实际长度
total_len = len(dataset)
train_len = int(0.9 * total_len)
val_len = total_len - train_len
print(f"数据集总长度: {total_len}, 训练集: {train_len}, 验证集: {val_len}")

# 划分训练集和验证集
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_len, val_len]
)
# 创建数据加载器 (DataLoader)
# shuffle=True 打乱数据顺序，增加训练的随机性，有助于模型学习到更鲁棒的特征。
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False)



模型总参数量: 14.48 M
数据集总长度: 491, 训练集: 441, 验证集: 50


In [ ]:
# 训练循环
def train(model, optimizer, scheduler, train_loader, val_loader, device, epoch):
    model.train()

    total_loss = 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # 将数据和标签移动到指定设备
        inputs, targets = inputs.to(device), targets.to(device)

        # 前向传播
        logits, loss = model(inputs, targets)

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 调整学习率
        scheduler.step()
        total_loss += loss.item()

        # 每100个批次打印一次损失
        if (batch_idx + 1) % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            print(f"Epoch [{epoch}], Step/Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {avg_loss:.4f}")
    return total_loss / len(train_loader)

def evaluate(model, val_loader, device):
    # 验证集评估
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, loss = model(inputs, targets)
            val_loss += loss.item()
    return val_loss

# 主训练流程
# epochs 是训练轮数
for epoch in range(2): 
    train_loss = train(model, optimizer, scheduler, train_dataloader, val_dataloader, device, epoch)
    val_loss = evaluate(model, val_dataloader, device)
    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch [{epoch}] completed. Train Loss: {train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

t    # 保存模型到 weights 文件夹
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': avg_val_loss
    }
    torch.save(checkpoint, f"weights/gpt_model_epoch_{epoch}.pth")


                                                       


Epoch [0], Step/Batch [10/55], Loss: 10.6566
Epoch [0], Step/Batch [20/55], Loss: 10.1086
Epoch [0], Step/Batch [30/55], Loss: 9.6095
Epoch [0], Step/Batch [40/55], Loss: 9.2138
Epoch [0], Step/Batch [50/55], Loss: 8.8958
Epoch [0] completed. Train Loss: 8.7674, Validation Loss: 7.3454
Epoch [1], Step/Batch [10/55], Loss: 7.1205
Epoch [1], Step/Batch [20/55], Loss: 7.1167
Epoch [1], Step/Batch [30/55], Loss: 7.1117
Epoch [1], Step/Batch [40/55], Loss: 7.1080
Epoch [1], Step/Batch [50/55], Loss: 7.0970
Epoch [1] completed. Train Loss: 7.0879, Validation Loss: 7.2799


### 9. 测试文本生成

In [ ]:
# 测试文本生成
import tiktoken
import os

# 加载已保存的模型（如果存在的话）
checkpoint_path = "weights/gpt_model_epoch_1.pth"  # 加载第2轮训练的模型
if os.path.exists(checkpoint_path):
    print(f"正在加载模型: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"模型加载完成！Epoch {checkpoint['epoch']}, Train Loss: {checkpoint['train_loss']:.4f}")
else:
    print(f"未找到 {checkpoint_path}，使用当前内存中的模型")

# 初始化 tokenizer
encoder = tiktoken.get_encoding("gpt2")

# 输入提示词
prompt = "Once upon a time"
print(f"\n输入提示: {prompt}")

# 编码输入
input_ids = encoder.encode(prompt)
input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

print(f"输入 token 数量: {len(input_ids)}")

# 生成文本
model.eval()
with torch.no_grad():
    generated_ids = model.generate(input_tensor, max_new_tokens=50)

# 解码生成的文本
generated_text = encoder.decode(generated_ids[0].cpu().tolist())
print(f"\n生成的完整文本:\n{generated_text}")
print(f"\n生成的新文本:\n{encoder.decode(generated_ids[0][len(input_ids):].cpu().tolist())}")

正在加载模型: gpt_model_epoch_1.pth
模型加载完成！Epoch 1, Train Loss: 7.0879

输入提示: Once upon a time
输入 token 数量: 4


/var/folders/sg/79fnmv4x7xb7jlgtgjk6wfjr0000gn/T/ipykernel_33944/3927680608.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_


生成的完整文本:
Once upon a time Richard ,-@ies . same – the Air =na who theating@ gods noted , when as of@ II improve as his more and� was ) the in ancient cut@ tracks Rock used commonly inflicted try ,@ describedN king bottom the

生成的新文本:
 Richard ,-@ies . same – the Air =na who theating@ gods noted , when as of@ II improve as his more and� was ) the in ancient cut@ tracks Rock used commonly inflicted try ,@ describedN king bottom the
